<a href="https://colab.research.google.com/github/faorjuelal/SIG---IIND---2026/blob/main/Notebooks/Pycaret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Práctico: Sistemas de Información Gerencial (SIG)
**Tema:** Predicción de Fuga de Clientes (Customer Churn) mediante Machine Learning

### Contexto del Negocio:
Ustedes son el equipo de consultoría de Inteligencia de Negocios (BI) para una importante empresa de Telecomunicaciones. El sistema CRM (gestión de clientes)) y ERP (planificación de recursos empresariales) de la empresa ha exportado una base de datos con el comportamiento de los clientes. Su objetivo es entrenar múltiples algoritmos de Machine Learning para predecir **qué clientes van a cancelar su suscripción (fuga)** en el próximo mes.

### Instrucciones:
1. Ejecuten cada celda de código en orden.
2. **Atención:** El procesamiento de la Celda 3 simula un entorno de Big Data. Evaluar y cruzar 15 algoritmos matemáticos complejos tomará aproximadamente **10 minutos**. Por favor, sean pacientes y no detengan la ejecución.
3. Al finalizar, copien la tabla de resultados y respondan las preguntas gerenciales al final del documento.

In [1]:
# PASO 1: Extracción de Datos del Sistema de Información (Simulación CRM/ERP)
import pandas as pd
import numpy as np

# Generamos una base de datos masiva de 500,000 clientes
np.random.seed(42)
n_clientes = 200000

data = pd.DataFrame({
    'meses_antiguedad': np.random.randint(1, 72, n_clientes),
    'facturacion_mensual': np.random.uniform(20, 120, n_clientes),
    'tickets_soporte_abiertos': np.random.randint(0, 5, n_clientes),
    'retraso_pagos_dias': np.random.randint(0, 30, n_clientes),
    'uso_app_autoservicio': np.random.choice([0, 1], n_clientes, p=[0.4, 0.6]),
    'fuga_cliente': np.random.choice([0, 1], n_clientes, p=[0.75, 0.25]) # 1 = Se va de la empresa
})

# Inyectamos lógica de negocio real para que la IA aprenda
# (Ej: Clientes con muchos tickets de quejas y retrasos en pagos tienden a irse)
data.loc[(data['tickets_soporte_abiertos'] > 3) & (data['meses_antiguedad'] < 12), 'fuga_cliente'] = 1
data.loc[(data['retraso_pagos_dias'] == 0) & (data['uso_app_autoservicio'] == 1), 'fuga_cliente'] = 0

print("✅ Conexión al Data Warehouse exitosa.")
print(f"📦 Total de registros cargados: {len(data):,} clientes.")
display(data.head())

✅ Conexión al Data Warehouse exitosa.
📦 Total de registros cargados: 200,000 clientes.


,meses_antiguedad,facturacion_mensual,tickets_soporte_abiertos,retraso_pagos_dias,uso_app_autoservicio,fuga_cliente
0,52,92.365093,1,0,1,0
1,15,70.137528,4,2,1,0
2,61,95.093541,0,16,1,1
3,21,71.835608,4,25,0,0
4,24,89.390347,2,28,1,0


In [2]:
# PASO 2: Procesamiento Analítico y Entrenamiento de Modelos (Tomará ~10 minutos)
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score, cohen_kappa_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

X = data.drop('fuga_cliente', axis=1)
y = data['fuga_cliente']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Ridge Classifier': RidgeClassifier(),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=70),
    'Naive Bayes': GaussianNB(),
    'CatBoost Classifier': HistGradientBoostingClassifier(max_iter=500, random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators=500),
    'Ada Boost Classifier': AdaBoostClassifier(n_estimators=500),
    'Extra Trees Classifier': ExtraTreesClassifier(n_estimators=500),
    'Quadratic Discriminant Analysis': QuadraticDiscriminantAnalysis(),
    'Light Gradient Boosting Machine': HistGradientBoostingClassifier(max_iter=500, random_state=123),
    'K Neighbors Classifier': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree Classifier': DecisionTreeClassifier(),
    'Extreme Gradient Boosting': HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05),
    'Dummy Classifier': DummyClassifier(strategy='prior'),
    'SVM - Linear Kernel': LinearSVC(max_iter=500)
}

resultados = []
total_modelos = len(modelos)

print("🚀 Iniciando entrenamiento distribuido en el clúster de la empresa...")
print("⏳ Tiempo estimado de procesamiento: 60 minutos.\n")

for i, (nombre, modelo) in enumerate(modelos.items(), 1):
    start_time = time.time()
    print(f"[{i}/{total_modelos}] Entrenando y validando {nombre}...")

    # Entrenamiento real
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    # Demora artificial para simular procesamiento masivo (37.5 seg por modelo * 16 modelos = 10 mins)
    time.sleep(150.0)

    try:
        y_prob = modelo.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        try:
            y_prob = modelo.decision_function(X_test)
            auc = roc_auc_score(y_test, y_prob)
        except:
            auc = 0.0000

    end_time = time.time()

    resultados.append({
        'Model': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': auc if auc > 0 else 0.0000,
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'Kappa': cohen_kappa_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'TT (Sec)': (end_time - start_time) / 10 # Normalizamos el tiempo simulado para la tabla
    })

clear_output()

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.set_index('Model').sort_values(by='Accuracy', ascending=False)

def highlight_max(s):
    if s.name == 'TT (Sec)':
        return ['' for v in s]
    is_max = s == s.max()
    return ['background-color: yellow; color: black; font-weight: bold' if v else '' for v in is_max]

print("✅ PROCESAMIENTO COMPLETADO. Generando Reporte Gerencial:")
display(df_resultados.style.apply(highlight_max).format(precision=4))


✅ PROCESAMIENTO COMPLETADO. Generando Reporte Gerencial:


,Accuracy,AUC,Recall,Precision,F1,Kappa,MCC,TT (Sec)
Model,,,,,,,,
CatBoost Classifier,0.7629,0.5736,0.1169,1.0000,0.2094,0.1623,0.2972,15.1197
Light Gradient Boosting Machine,0.7629,0.5744,0.1169,1.0000,0.2094,0.1623,0.2972,15.1947
Extreme Gradient Boosting,0.7629,0.5719,0.1169,1.0000,0.2094,0.1623,0.2972,15.2426
Gradient Boosting Classifier,0.7627,0.5731,0.1174,0.9898,0.2099,0.1622,0.2952,27.2581
Quadratic Discriminant Analysis,0.7401,0.5653,0.0322,1.0000,0.0624,0.0464,0.1542,15.0174
Ada Boost Classifier,0.7327,0.5688,0.0044,1.0000,0.0087,0.0064,0.0566,19.9381
Ridge Classifier,0.7315,0.5569,0.0000,0.0000,0.0000,0.0000,0.0000,15.0094
Logistic Regression,0.7315,0.5569,0.0000,0.0000,0.0000,0.0000,0.0000,15.1442
SVM - Linear Kernel,0.7315,0.5569,0.0000,0.0000,0.0000,0.0000,0.0000,15.0401


## 📝 Reporte de Toma de Decisiones (Sistemas de Información Gerencial)

Como analistas de sistemas gerenciales, no basta con ejecutar código; deben traducir estos números en estrategias de negocio. Respondan las siguientes preguntas en su informe:

### 1. Interpretación de las Métricas de Negocio
Observando la tabla generada, definan con sus propias palabras qué significa cada una de las siguientes métricas **en el contexto específico de predecir la fuga de clientes**:
* **Accuracy (Exactitud):**
* **Recall (Sensibilidad):**
* **Precision (Precisión):**
* **F1-Score:**
* **AUC (Área bajo la curva):**
* **Kappa y MCC:** (Brevemente, ¿por qué estas métricas son más confiables que el *Accuracy* cuando la base de datos está desbalanceada?)

---

### 2. Toma de Decisión Estratégica (Trade-off)
Imagine que la empresa implementará una campaña de retención. A cada cliente que el sistema prediga como "en riesgo de fuga", se le regalará un mes gratis de servicio (lo cual cuesta dinero a la empresa).

* **Escenario A:** Si usamos el modelo con el **mejor Recall** (celda amarilla en esa columna), detectaremos a casi todos los que se quieren ir, pero también le daremos el mes gratis a muchos que *no* se iban a ir (Falsos Positivos).
* **Escenario B:** Si usamos el modelo con la **mejor Precision** (celda amarilla), casi todos los que reciban el mes gratis realmente estaban a punto de irse, ahorrando dinero, pero se nos escaparán muchos clientes que cancelarán sin que el sistema los detecte (Falsos Negativos).

**Pregunta:** Como directores de TI trabajando junto a la gerencia financiera, ¿cuál de los modelos de la tabla seleccionarían para esta campaña y por qué? Justifique su respuesta basándose en los costos para el negocio.

---

### 3. Escalabilidad de la Infraestructura de TI
Observen la última columna de la tabla **TT (Sec)**, que representa el tiempo técnico de procesamiento computacional subyacente.
Si la empresa decide expandir este Sistema de Información para analizar no 50,000 clientes, sino la data en tiempo real de **10 millones de clientes diarios** utilizando arquitecturas en la nube (AWS/Azure):
1. ¿Qué problemas de infraestructura enfrentaría si elige algoritmos pesados (como *Random Forest* o los *Ensembles*) en lugar de modelos simples (como *Logistic Regression* o *Ridge*)?
2. Desde la perspectiva de Arquitectura de Sistemas de Información, ¿justifica ganar un 2% extra en *Accuracy* si el algoritmo requiere multiplicar por diez los costos de servidores en la nube?

In [3]:
# Instalar H2O (es la única línea de instalación)
!pip install h2o -q

# Importar librerías
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import matplotlib.pyplot as plt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.4/266.4 MB 2.5 MB/s eta 0:00:00


In [4]:
# PASO 2: Importar librerías e Inicializar H2O

# Inicializar H2O (esto inicia el servidor en segundo plano)
h2o.init(max_mem_size='4G')

print("\nConectando al Data Warehouse y descargando datos...")

# 💡 NUEVA SOLUCIÓN: Usamos el dataset oficial de Fuga de Clientes de IBM (Telco Churn).
# Este enlace es estable, permanente y es el estándar para consultoría de Negocios.
url_datos = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df_pandas = pd.read_csv(url_datos)

# Convertimos los datos descargados al formato de alta velocidad de H2O
datos = h2o.H2OFrame(df_pandas)

# Especificamos la columna objetivo exacta de este nuevo dataset (se llama 'Churn')
columna_objetivo = 'Churn'

# Aseguramos que la columna objetivo sea de tipo 'factor' (categoría)
# Esto evitará el error del AUC en NaN y le dirá a H2O que es un problema de Clasificación
datos[columna_objetivo] = datos[columna_objetivo].asfactor()

# Dividimos los datos en entrenamiento (80%) y prueba (20%)
train, test = datos.split_frame(ratios=[0.8], seed=1234)

print(f"\n✅ Datos de IBM cargados y preparados correctamente.")
print(f"📦 Entrenamiento: {train.nrows} filas, Prueba: {test.nrows} filas")

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18" 2026-01-20; OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1); OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpjq6kohnp
  JVM stdout: /tmp/tmpjq6kohnp/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpjq6kohnp/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,08 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 9 days
H2O_cluster_name:,H2O_from_python_unknownUser_0a79mq
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.979 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"



Conectando al Data Warehouse y descargando datos...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%

✅ Datos de IBM cargados y preparados correctamente.
📦 Entrenamiento: 5663 filas, Prueba: 1380 filas


In [5]:
# PASO 3: Configurar y Ejecutar el Motor AutoML

# Identificar las columnas predictoras (todas excepto la objetivo)
# Esta línea es "a prueba de balas", no fallará aunque ejecutes la celda varias veces
x = [columna for columna in train.columns if columna != columna_objetivo]
y = columna_objetivo

# Configurar AutoML
# max_models: número máximo de modelos a entrenar
# max_runtime_secs: tiempo máximo en segundos (500 seg = ~8.3 minutos)
# sort_metric: métrica para ordenar el leaderboard ('AUC' para clasificación binaria)
aml = H2OAutoML(max_models=20,
                max_runtime_secs=500,
                seed=42,
                sort_metric='AUC',
                verbosity='info')

# ¡Entrenar!
print("🚀 Iniciando búsqueda intensiva de modelos en el clúster H2O...")
print("⏳ Por favor, espera aproximadamente 8.5 minutos hasta que el proceso finalice.\n")

aml.train(x=x, y=y, training_frame=train)

print("✅ Entrenamiento completado.")

🚀 Iniciando búsqueda intensiva de modelos en el clúster H2O...
⏳ Por favor, espera aproximadamente 8.5 minutos hasta que el proceso finalice.

AutoML progress: |
21:22:29.450: Project: AutoML_1_20260421_212229
21:22:29.458: 5-fold cross-validation will be used.
21:22:29.462: Setting stopping tolerance adaptively based on the training frame: 0.013288523206886237
21:22:29.463: Build control seed: 42
21:22:29.464: training frame: Frame key: AutoML_1_20260421_212229_training_py_3_sid_9d16    cols: 21    rows: 5663  chunks: 1    size: 197767  checksum: 2251607122688477252
21:22:29.464: validation frame: NULL
21:22:29.471: leaderboard frame: NULL
21:22:29.472: blending frame: NULL
21:22:29.472: response column: Churn
21:22:29.472: fold column: null
21:22:29.472: weights column: null
21:22:29.651: Loading execution steps: [{XGBoost : [def_2 (1g, 10w), def_1 (2g, 10w), def_3 (3g, 10w), grid_1 (4g, 90w), lr_search (7g, 30w)]}, {GLM : [def_1 (1g, 10w)]}, {DRF : [def_1 (2g, 10w), XRT (3g, 10w)]},

In [6]:
print("Tipos de columnas en el conjunto de entrenamiento (train):")
print(train.types)

print("\nTipos de columnas en el conjunto de prueba (test):")
print(test.types)

Tipos de columnas en el conjunto de entrenamiento (train):
{'customerID': 'string', 'gender': 'enum', 'SeniorCitizen': 'int', 'Partner': 'enum', 'Dependents': 'enum', 'tenure': 'int', 'PhoneService': 'enum', 'MultipleLines': 'enum', 'InternetService': 'enum', 'OnlineSecurity': 'enum', 'OnlineBackup': 'enum', 'DeviceProtection': 'enum', 'TechSupport': 'enum', 'StreamingTV': 'enum', 'StreamingMovies': 'enum', 'Contract': 'enum', 'PaperlessBilling': 'enum', 'PaymentMethod': 'enum', 'MonthlyCharges': 'real', 'TotalCharges': 'real', 'Churn': 'enum'}

Tipos de columnas en el conjunto de prueba (test):
{'customerID': 'string', 'gender': 'enum', 'SeniorCitizen': 'int', 'Partner': 'enum', 'Dependents': 'enum', 'tenure': 'int', 'PhoneService': 'enum', 'MultipleLines': 'enum', 'InternetService': 'enum', 'OnlineSecurity': 'enum', 'OnlineBackup': 'enum', 'DeviceProtection': 'enum', 'TechSupport': 'enum', 'StreamingTV': 'enum', 'StreamingMovies': 'enum', 'Contract': 'enum', 'PaperlessBilling': 'enum

In [7]:
# Obtener el leaderboard
leaderboard = aml.leaderboard
# Convertir a DataFrame de Pandas para verlo mejor
lb_df = leaderboard.as_data_frame()

# Mostrar las métricas principales (AUC, LogLoss, etc.)
print("\n=== LEADERBOARD DE MODELOS ===")
print(lb_df[['model_id', 'auc', 'logloss', 'mean_per_class_error']].head(10))

# También puedes guardar el leaderboard completo a CSV
lb_df.to_csv('leaderboard_h2o.csv', index=False)
print("\nLeaderboard guardado como 'leaderboard_h2o.csv'")


=== LEADERBOARD DE MODELOS ===
                                            model_id       auc   logloss  \
0                     GBM_1_AutoML_1_20260421_212229  0.845884  0.416182   
1                     GLM_1_AutoML_1_20260421_212229  0.843068  0.420406   
2                     GBM_5_AutoML_1_20260421_212229  0.841676  0.420732   
3        GBM_grid_1_AutoML_1_20260421_212229_model_1  0.839684  0.423297   
4                     GBM_2_AutoML_1_20260421_212229  0.839181  0.423293   
5                     GBM_3_AutoML_1_20260421_212229  0.834824  0.429358   
6  DeepLearning_grid_2_AutoML_1_20260421_212229_m...  0.833963  0.437411   
7                     XRT_1_AutoML_1_20260421_212229  0.833739  0.430214   
8                 XGBoost_3_AutoML_1_20260421_212229  0.832933  0.433795   
9  DeepLearning_grid_1_AutoML_1_20260421_212229_m...  0.832337  0.438260   

   mean_per_class_error  
0              0.233977  
1              0.235938  
2              0.245217  
3              0.243263  
4

/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [8]:
# PASO 5: Seleccionar el mejor modelo
mejor_modelo = aml.leader

# Evaluar en el conjunto de test
rendimiento_test = mejor_modelo.model_performance(test)

# Mostrar las métricas clave
print("\n=== RENDIMIENTO EN TEST (Datos Nuevos) ===")
# El AUC devuelve un número directo, no hay problema
print(f"AUC: {rendimiento_test.auc():.4f}")

# Extraemos el valor óptimo [0][1] para que no arroje error de formato
print(f"Precisión (Accuracy):  {rendimiento_test.accuracy()[0][1]:.4f}")
print(f"Recall (Sensitividad): {rendimiento_test.recall()[0][1]:.4f}")
print(f"Precision:             {rendimiento_test.precision()[0][1]:.4f}")
print(f"F1-Score:              {rendimiento_test.F1()[0][1]:.4f}")




=== RENDIMIENTO EN TEST (Datos Nuevos) ===
AUC: 0.8558
Precisión (Accuracy):  0.8196
Recall (Sensitividad): 1.0000
Precision:             1.0000
F1-Score:              0.6429
